# GloVe and FastText Word Embeddings

In this notebook we explore two popular pre-trained word embedding approaches:
- **GloVe** (Global Vectors for Word Representation): learns from global word co-occurrence statistics
- **FastText**: extends Word2Vec by also learning subword (character n-gram) representations

We cover: loading vectors, handling OOV words, computing similarity, and solving analogy tasks.

In [ ]:
# Install required libraries
!pip install gensim numpy -q

import numpy as np
from gensim.models import KeyedVectors, Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.8 MB/s eta 0:00:00


## Step 1: Build Demo Vectors

Downloading the real GloVe model (400MB+) is slow in notebook environments. We build compact demo vectors with hand-crafted semantic axes so all operations work correctly offline.

In [ ]:
def build_demo_glove():
    dim = 50
    # First 5 dimensions encode semantic meaning manually:
    # [royalty, gender, animal, geography, sentiment]
    def vec(royalty, gender, animal, geo, sentiment, seed):
        np.random.seed(seed)
        # Fill all 50 dims with small random noise
        v = np.random.normal(0, 0.05, dim).astype(np.float32)
        # Override first 5 dims with meaningful semantic values
        v[0], v[1], v[2], v[3], v[4] = royalty, gender, animal, geo, sentiment
        return v

    # Each word gets a vector with semantic encoding + random noise
    words = {
        "king"    : vec( 0.9,  0.8, -0.1,  0.0,  0.0,  1),
        "queen"   : vec( 0.9, -0.8, -0.1,  0.0,  0.0,  2),
        "prince"  : vec( 0.7,  0.7, -0.1,  0.0,  0.1,  3),
        "princess": vec( 0.7, -0.7, -0.1,  0.0,  0.1,  4),
        "man"     : vec( 0.0,  0.9,  0.0,  0.0,  0.0,  5),
        "woman"   : vec( 0.0, -0.9,  0.0,  0.0,  0.0,  6),
        "boy"     : vec( 0.0,  0.7,  0.0,  0.0,  0.2,  7),
        "girl"    : vec( 0.0, -0.7,  0.0,  0.0,  0.2,  8),
        "cat"     : vec(-0.2,  0.0,  0.9,  0.0,  0.0,  9),
        "dog"     : vec(-0.2,  0.0,  0.8,  0.0,  0.1, 10),
        "lion"    : vec(-0.1,  0.3,  0.9,  0.0, -0.1, 11),
        "tiger"   : vec(-0.1,  0.1,  0.85, 0.0, -0.1, 12),
        "paris"   : vec( 0.0,  0.0, -0.1,  0.9,  0.1, 13),
        "france"  : vec( 0.0,  0.0, -0.1,  0.8,  0.1, 14),
        "berlin"  : vec( 0.0,  0.0, -0.1,  0.85,-0.1, 15),
        "germany" : vec( 0.0,  0.0, -0.1,  0.75,-0.1, 16),
        "london"  : vec( 0.0,  0.0, -0.1,  0.8,  0.0, 17),
        "england" : vec( 0.0,  0.0, -0.1,  0.7,  0.0, 18),
        "good"    : vec( 0.0,  0.0,  0.0,  0.0,  0.9, 19),
        "better"  : vec( 0.0,  0.0,  0.0,  0.0,  0.95,20),
        "bad"     : vec( 0.0,  0.0,  0.0,  0.0, -0.9, 21),
        "worse"   : vec( 0.0,  0.0,  0.0,  0.0, -0.95,22),
        "banana"  : vec(-0.5, -0.2,  0.0,  0.0,  0.3, 23),
        "bird"    : vec(-0.1,  0.0,  0.7,  0.0,  0.1, 24),
        "fish"    : vec(-0.1,  0.0,  0.6,  0.0,  0.0, 25),
    }
    # Build a gensim KeyedVectors object from our word-vector dictionary
    kv = KeyedVectors(vector_size=dim)
    kv.add_vectors(list(words.keys()), list(words.values()))
    return kv


def build_demo_fasttext():
    # We use Word2Vec as a stand-in for FastText in the demo
    # In real FastText, subword character n-grams are also used
    sentences = [
        "the cat sat on the mat".split(),
        "dogs are loyal animals".split(),
        "the king rules the kingdom".split(),
        "the queen rules beside the king".split(),
        "a man walked into the forest".split(),
        "the woman followed the man".split(),
        "the lion is the king of the jungle".split(),
        "animals live in different habitats".split(),
        "good better bad worse".split(),
        "paris is the capital of france".split(),
        "berlin is the capital of germany".split(),
        "man king woman queen prince princess".split(),
    ]
    # Train Word2Vec with sg=1 (Skip-Gram) as FastText substitute
    model = Word2Vec(sentences, vector_size=50, window=3,
                     min_count=1, sg=1, epochs=300, seed=42)
    return model.wv

print("Demo builder functions defined.")

Demo builder functions defined.


## Step 2: Load GloVe

We first try the real pre-trained GloVe model via gensim downloader. If not available (offline/slow), we fall back to our demo vectors.

In [ ]:
def load_glove():
    try:
        import gensim.downloader as api
        # This downloads ~66MB of 50-dim GloVe vectors trained on Wikipedia
        model = api.load("glove-wiki-gigaword-50")
        return model, False   # False = not demo
    except Exception:
        # Fall back to demo vectors if download fails
        return build_demo_glove(), True

glove, is_demo = load_glove()
mode = "DEMO (offline)" if is_demo else "REAL (glove-wiki-gigaword-50)"
print(f"GloVe loaded [{mode}]")
print(f"Vocabulary size : {len(glove):,}")
print(f"Vector dimensions: {glove.vector_size}")

[==================================================] 100.0% 66.0/66.0MB downloaded
GloVe loaded [REAL (glove-wiki-gigaword-50)]
Vocabulary size : 400,000
Vector dimensions: 50


## Step 3: Convert Words to Vectors

Each word maps to a dense float vector. The numbers themselves are not meaningful individually — only the distances and directions between vectors matter.

In [ ]:
# Display vectors for 5 words (showing only first 8 values for readability)
print(f"{'Word':<10}  First 8 vector values")
print("-" * 55)
for word in ['king', 'queen', 'man', 'woman', 'cat']:
    vec = glove[word]   # lookup the vector for this word
    vals = ', '.join([f'{v:.3f}' for v in vec[:8]])
    print(f"{word:<10}  [{vals} ...]")

Word        First 8 vector values
-------------------------------------------------------
king        [0.505, 0.686, -0.595, -0.023, 0.600, -0.135, -0.088, 0.474 ...]
queen       [0.379, 1.823, -1.265, -0.104, 0.358, 0.600, -0.175, 0.838 ...]
man         [-0.094, 0.430, -0.172, -0.455, 1.645, 0.403, -0.373, 0.251 ...]
woman       [-0.182, 0.648, -0.582, -0.495, 1.541, 1.345, -0.433, 0.581 ...]
cat         [0.453, -0.501, -0.537, -0.016, 0.222, 0.546, -0.673, -0.689 ...]


## Step 4: GloVe OOV (Out-of-Vocabulary) Handling

GloVe stores only words seen during training. Any new word (like 'ChatGPT') will cause a KeyError — it has no vector.

In [ ]:
def get_glove_vector(model, word):
    w = word.lower()
    # Check if word exists in GloVe vocabulary
    if w in model:
        return model[w]
    else:
        return None  # OOV: word not found

# Test with known words and unknown (OOV) words
test_words = ['king', 'smartphone', 'ChatGPT', 'dog', 'xyzabc123']

print("GloVe OOV Test:")
print(f"  {'Word':<18}  Status")
print(f"  {'-'*18}  {'-'*30}")
for w in test_words:
    v = get_glove_vector(glove, w)
    if v is not None:
        status = f"FOUND  (v[0] = {v[0]:.4f})"
    else:
        status = "OOV — no vector available"
    print(f"  {w:<18}  {status}")

GloVe OOV Test:
  Word                Status
  ------------------  ------------------------------
  king                FOUND  (v[0] = 0.5045)
  smartphone          FOUND  (v[0] = 0.6631)
  ChatGPT             OOV — no vector available
  dog                 FOUND  (v[0] = 0.1101)
  xyzabc123           OOV — no vector available


## Step 5: GloVe Word Similarity

Cosine similarity measures the angle between two vectors. Score of 1.0 = identical direction, 0.0 = unrelated, -1.0 = opposite meaning.

In [ ]:
# Word pairs to compare for similarity
pairs = [
    ('king',  'queen'),     # both royalty — should be high
    ('man',   'woman'),     # both human, different gender
    ('cat',   'dog'),       # both domestic animals
    ('paris', 'france'),    # capital and country
    ('good',  'bad'),       # antonyms — should be negative or low
    ('king',  'banana'),    # completely unrelated — should be very low
]

print(f"  {'Pair':<22}  {'Similarity':>10}")
print(f"  {'-'*22}  {'-'*10}")
for w1, w2 in pairs:
    try:
        score = glove.similarity(w1, w2)
        print(f"  {w1 + ' / ' + w2:<22}  {score:>10.4f}")
    except KeyError as e:
        print(f"  {w1 + ' / ' + w2:<22}  not in vocab: {e}")

  Pair                    Similarity
  ----------------------  ----------
  king / queen                0.7839
  man / woman                 0.8860
  cat / dog                   0.9218
  paris / france              0.8025
  good / bad                  0.7965
  king / banana               0.2207


## Step 6: GloVe Analogy Tasks

Analogie formula: `b - a + c ≈ d`  
Example: `king - man + woman ≈ queen`

We use `most_similar(positive=[b, c], negative=[a])` to find `d`.

In [ ]:
def solve_analogy(model, a, b, c, label=""):
    tag = f"[{label}] " if label else ""
    print(f"  {tag}'{a}' : '{b}'  ::  '{c}' : ?")
    try:
        # Vector math: result ≈ b - a + c
        results = model.most_similar(positive=[b, c], negative=[a], topn=3)
        for rank, (word, score) in enumerate(results, 1):
            print(f"    {rank}. {word:<14}  score: {score:.4f}")
    except KeyError as e:
        print(f"    Word not in vocab: {e}")
    print()

# Classic analogy tasks
analogies = [
    ('man',   'king',   'woman'),   # expected answer: queen
    ('paris', 'france', 'berlin'),  # expected answer: germany
    ('good',  'better', 'bad'),     # expected answer: worse
    ('king',  'prince', 'queen'),   # expected answer: princess
]

print("GloVe Analogy Results:")
print("-" * 40)
for a, b, c in analogies:
    solve_analogy(glove, a, b, c, "GloVe")

GloVe Analogy Results:
----------------------------------------
  [GloVe] 'man' : 'king'  ::  'woman' : ?
    1. queen           score: 0.8524
    2. throne          score: 0.7664
    3. prince          score: 0.7592

  [GloVe] 'paris' : 'france'  ::  'berlin' : ?
    1. germany         score: 0.9212
    2. denmark         score: 0.8076
    3. poland          score: 0.7936

  [GloVe] 'good' : 'better'  ::  'bad' : ?
    1. worse           score: 0.9011
    2. too             score: 0.8324
    3. unfortunately   score: 0.8224

  [GloVe] 'king' : 'prince'  ::  'queen' : ?
    1. princess        score: 0.8546
    2. duchess         score: 0.7672
    3. victoria        score: 0.7387



## Step 7: Load FastText

FastText extends Word2Vec by also learning character n-gram embeddings. This means it can generate vectors for **unseen words** using their subword parts (e.g., 'playing' → 'play', 'lay', 'aying').

In [ ]:
def load_fasttext():
    try:
        import gensim.downloader as api
        # Real FastText model trained on Wikipedia (300 dims, ~900MB)
        model = api.load("fasttext-wiki-news-subwords-300")
        return model, False
    except Exception:
        return build_demo_fasttext(), True

ft, ft_demo = load_fasttext()
mode = "DEMO" if ft_demo else "REAL (fasttext-wiki-news-subwords-300)"
print(f"FastText loaded [{mode}]")
print(f"Vocabulary size  : {len(ft):,}")
print(f"Vector dimensions: {ft.vector_size}")

[==================================================] 100.0% 958.5/958.4MB downloaded
FastText loaded [REAL (fasttext-wiki-news-subwords-300)]
Vocabulary size  : 999,999
Vector dimensions: 300


## Step 8: FastText OOV via Character N-grams

FastText represents every word as a bag of character n-grams. Even a word not seen during training can get a vector by combining its subword pieces.

In [ ]:
def get_character_ngrams(word, min_n=3, max_n=6):
    # Add boundary markers < > around the word (FastText convention)
    padded = f"<{word}>"
    ngrams = []
    for n in range(min_n, max_n + 1):
        # Slide a window of size n across the padded word
        for i in range(len(padded) - n + 1):
            ngrams.append(padded[i:i+n])
    return ngrams

def get_fasttext_vector(model, word):
    w = word.lower()
    if w in model:
        return model[w], "exact match"
    # If word is OOV, average the vectors of known n-grams
    ngrams = get_character_ngrams(w)
    known_ngrams = [ng for ng in ngrams if ng in model]
    if known_ngrams:
        vecs = np.array([model[ng] for ng in known_ngrams])
        return vecs.mean(axis=0).astype(np.float32), f"n-gram avg ({len(known_ngrams)}/{len(ngrams)} found)"
    # If no n-grams known either, return zero vector
    return np.zeros(model.vector_size, dtype=np.float32), "zero fallback (completely unknown)"

# Show what character n-grams look like
print("Character n-grams for the word 'playing':")
print(" ", get_character_ngrams("playing"))
print()

# Compare GloVe vs FastText OOV coverage
oov_test_words = ['king', 'animals', 'unhappiness', 'playfulness',
                  'ChatGPT', 'xyzabc123', 'running', 'walked']

print(f"  {'Word':<22}  {'GloVe':<14}  FastText")
print(f"  {'-'*22}  {'-'*12}  {'-'*35}")
for w in oov_test_words:
    g_status = "found" if get_glove_vector(glove, w) is not None else "OOV"
    _, f_method = get_fasttext_vector(ft, w)
    print(f"  {w:<22}  {g_status:<14}  {f_method}")

Character n-grams for the word 'playing':
  ['<pl', 'pla', 'lay', 'ayi', 'yin', 'ing', 'ng>', '<pla', 'play', 'layi', 'ayin', 'ying', 'ing>', '<play', 'playi', 'layin', 'aying', 'ying>', '<playi', 'playin', 'laying', 'aying>']

  Word                    GloVe           FastText
  ----------------------  ------------  -----------------------------------
  king                    found           exact match
  animals                 found           exact match
  unhappiness             found           exact match
  playfulness             found           exact match
  ChatGPT                 OOV             n-gram avg (4/22 found)
  xyzabc123               OOV             n-gram avg (7/30 found)
  running                 found           exact match
  walked                  found           exact match


## Step 9: Similarity Comparison — GloVe vs FastText

In [ ]:
def cosine_similarity(v1, v2):
    # Cosine similarity = dot product / (magnitude1 * magnitude2)
    n1, n2 = np.linalg.norm(v1), np.linalg.norm(v2)
    if n1 == 0 or n2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (n1 * n2))

print(f"  {'Pair':<22}  {'GloVe':>8}  {'FastText':>8}")
print(f"  {'-'*22}  {'-'*8}  {'-'*8}")
for w1, w2 in pairs:
    try:
        g_sim = glove.similarity(w1, w2)
    except KeyError:
        g_sim = None
    v1_vec, _ = get_fasttext_vector(ft, w1)
    v2_vec, _ = get_fasttext_vector(ft, w2)
    f_sim = cosine_similarity(v1_vec, v2_vec)
    g_str = f"{g_sim:8.4f}" if g_sim is not None else "     N/A"
    print(f"  {w1 + ' / ' + w2:<22}  {g_str}  {f_sim:8.4f}")

  Pair                       GloVe  FastText
  ----------------------  --------  --------
  king / queen              0.7839    0.7704
  man / woman               0.8860    0.8120
  cat / dog                 0.9218    0.7502
  paris / france            0.8025    0.6667
  good / bad                0.7965    0.8503
  king / banana             0.2207    0.3231


## Step 10: FastText Analogies

In [ ]:
print("FastText Analogy Results:")
print("-" * 40)
for a, b, c in analogies:
    solve_analogy(ft, a, b, c, "FastText")

FastText Analogy Results:
----------------------------------------
  [FastText] 'man' : 'king'  ::  'woman' : ?
    1. queen           score: 0.7787
    2. queen-mother    score: 0.7144
    3. king-           score: 0.6981

  [FastText] 'paris' : 'france'  ::  'berlin' : ?
    1. germany         score: 0.6788
    2. poland          score: 0.6069
    3. europe          score: 0.6013

  [FastText] 'good' : 'better'  ::  'bad' : ?
    1. worse           score: 0.8504
    2. bettter         score: 0.7247
    3. worse-          score: 0.7173

  [FastText] 'king' : 'prince'  ::  'queen' : ?
    1. princess        score: 0.8220
    2. duchess         score: 0.7203
    3. princesse       score: 0.6923



## Conclusion

| Feature | GloVe | FastText |
|---------|-------|----------|
| Training basis | Global co-occurrence matrix | Local context + subword n-grams |
| OOV words | Cannot handle — KeyError | Handles via character n-grams |
| Vector quality | Strong on large corpora | Better for morphologically rich languages |
| Model size | Smaller | Larger (stores subword info) |

FastText's biggest advantage is **OOV handling** — it can produce meaningful vectors for words it has never seen by decomposing them into character pieces.